# Chapter 07: Multi-Armed Bandits, Gittins & Whittle Indices
### *Restless Bandits, Exploration vs. Exploitation, and the "Quarantine Blindness" Problem*

**[Course Home](../README.md) | [Python Lab](../python/07_multi_armed_bandits_gittins_and_whittle_indices.ipynb) | [Prev: Ch 06](06_optimal_stopping_and_hysteresis_control.ipynb) | [Next: Ch 08](08_quickest_change_detection_cusum_and_shiryaev_roberts.ipynb)**

---

## 1. Motivation: Why Rested Bandits Fail in Physical & Distributed Infrastructure

A canonical problem in computing systems is the optimal allocation of finite observation or processing capacity across competing stochastic processes:
* **CI Test & Runner Scheduling**: Allocating $M = 10$ soak-test worker slots across $K = 200$ flaky test targets or bare-metal testbeds.
* **Server Health Probing**: Selecting which remote microservices or database replicas to probe with active synthetic heartbeats.
* **Traffic Routing**: Splitting network packets across multiple heterogeneous egress paths with time-varying congestion.

### The Classical Rested Bandit Assumption
In the classical Multi-Armed Bandit (MAB) framework formalized by Robbins (1952) and solved by Gittins (1979):
1. The operator faces $K$ independent arms.
2. At each time step $t$, the operator pulls an arm $k \in \{1, \dots, K\}$.
3. The pulled arm yields a stochastic reward $R_k(t)$ and transitions to a new Markovian state according to a transition kernel $P_k$.
4. **The Rested Assumption**: All unpulled (passive) arms remain frozen in time—their states do not evolve, and their underlying distributions do not change.

Under this rested assumption, Gittins proved a celebrated result: the $K$-dimensional Markov Decision Process decomposes into $K$ independent 1-dimensional scalar optimization problems. The optimal policy is greedy with respect to the **Gittins Index** $\nu_k(s_k)$, which measures the maximum expected discounted reward per unit of expected discounted time.

### The Failure of the Rested Model: The "Restless" Reality
In physical engineering and software infrastructure, **arms are almost never rested**:
* A quarantined CI test does not remain frozen. Developers merge new code, refactor shared libraries, and deploy runtime dependency patches to the main branch. A flaky test that failed yesterday due to a database connection leak may have been silently fixed by a pull request merged four hours ago.
* A degraded cloud worker node or hardware testbed with elevated thermal throttling cools down over time or completes background garbage collection when idle.
* An unprobed network link clears its queue congestion or experiences a fiber cut while unobserved.

When passive arms continue to transition according to autonomous dynamics, the system is a **Restless Multi-Armed Bandit (RMAB)**, formulated by Peter Whittle (1988).

### The "Quarantine Blindness" Problem
The most severe manifestation of naive scheduling in production engineering is **Quarantine Blindness**:
When an automated monitor detects that a component, test, or server has degraded, it demotes the entity to a passive quarantine pool to protect the primary merge queue or production user traffic ($a = 0$).
* In the quarantine pool, the observation rate drops to zero ($p_{\text{obs}} = 0$).
* If the underlying component self-heals or is fixed by a background commit (transitioning from faulty to healthy with probability $q_{01} > 0$), the passive monitor never receives any evidence of the recovery.
* Because the system receives zero new observations, the posterior belief remains stuck at high degradation probability, and the entity remains quarantined indefinitely.

```
 +------------------+     High Flake / Failure      +------------------+
 |   Active Gate    | ----------------------------> | Quarantine Pool  |
 | (p_obs = 100%)   |                               |  (p_obs = 0%)    |
 +------------------+                               +------------------+
          ^                                                   |
          |                                                   |
          |       Can the system ever discover healing?       |
          + - - - - - - - - - - - - - - - - - - - - - - - - - +
                 NO! (Observation Rate is Zero!)
                 QUARANTINE BLINDNESS = INFINITE TRAP
```

To break this trap without wasting the entire worker pool on speculative runs, the scheduling policy must solve a restless POMDP. The mathematical tool for this allocation is the **Whittle Index**.

---

## 2. Mathematical Formulation of Restless Bandits

### 2.1 The Restless MDP Setup
Consider $K$ independent stochastic processes (arms). Each arm $k \in \{1, \dots, K\}$ has:
* A state space $\mathcal{S}_k$.
* An action space $\mathcal{A} = \{0, 1\}$, where $a_k = 1$ denotes pulling the arm (active: executing a test, probing a node) and $a_k = 0$ denotes leaving the arm idle (passive: keeping in quarantine, not probing).
* Transition kernels conditioned on the action:
  * If $a_k = 1$: state transitions according to $P_k^{(1)}(s' \mid s)$.
  * If $a_k = 0$: state transitions according to $P_k^{(0)}(s' \mid s)$.
* Instantaneous reward $R_k(s, a)$.

At each time step $t \in \{0, 1, 2, \dots\}$, the operator must choose a joint action vector $\mathbf{a}_t = (a_{1, t}, \dots, a_{K, t}) \in \{0, 1\}^K$ subject to a strict capacity budget of $M < K$ concurrent active arms:

$$
\sum_{k=1}^K a_{k, t} \le M, \quad \forall t
$$

The infinite-horizon discounted reward objective with discount factor $\gamma \in (0, 1)$ is:

$$
\max_{\pi} \mathbb{E}^\pi \left[ \sum_{t=0}^\infty \gamma^t \sum_{k=1}^K R_k(S_{k, t}, a_{k, t}) \right]
$$

### 2.2 Complexity: The Curse of Dimensionality and PSPACE-Hardness
Unlike the rested bandit where the Gittins index yields the exact optimal solution, restless bandits are notoriously difficult.
* The state space of the combined system is the Cartesian product $\mathcal{S}_1 \times \mathcal{S}_2 \times \dots \times \mathcal{S}_K$. If each arm has $N$ states, the global state space has cardinality $N^K$.
* Papadimitriou and Tsitsiklis (1999) proved that solving the restless multi-armed bandit problem to optimality is **PSPACE-hard**, even when transition probabilities and rewards are deterministic and known!

To obtain a computationally tractable policy that scales to hundreds or thousands of arms, we turn to **Whittle's Lagrangian Relaxation**.

---

## 3. Whittle's Lagrangian Relaxation and Indexability

### 3.1 Relaxing the Hard Budget Constraint
Whittle (1988) proposed replacing the strict instantaneous capacity constraint ($\sum_{k=1}^K a_{k, t} \le M$ for every $t$) with a time-average expected constraint:

$$
\mathbb{E} \left[ \sum_{t=0}^\infty \gamma^t \sum_{k=1}^K a_{k, t} \right] \le \frac{M}{1 - \gamma}
$$

Formulating the Lagrangian with Lagrange multiplier $\lambda \ge 0$:

$$
\mathcal{L}(\pi, \lambda) = \mathbb{E}^\pi \left[ \sum_{t=0}^\infty \gamma^t \sum_{k=1}^K R_k(S_{k, t}, a_{k, t}) \right] + \lambda \left( \frac{M}{1 - \gamma} - \mathbb{E}^\pi \left[ \sum_{t=0}^\infty \gamma^t \sum_{k=1}^K a_{k, t} \right] \right)
$$

Rearranging the terms:

$$
\mathcal{L}(\pi, \lambda) = \frac{\lambda M}{1 - \gamma} + \sum_{k=1}^K \mathbb{E}^\pi \left[ \sum_{t=0}^\infty \gamma^t \left( R_k(S_{k, t}, a_{k, t}) - \lambda a_{k, t} \right) \right]
$$

Notice the transformation: The constraint linking the $K$ arms has vanished! The objective has decoupled into $K$ completely independent single-arm subproblems, plus an additive constant.

### 3.2 The Subsidy for Passivity
We can interpret $\lambda$ as a **subsidy for passivity**:
* If you pull arm $k$ ($a_k = 1$), you earn the operational reward $R_k(s, 1)$.
* If you leave arm $k$ passive ($a_k = 0$), you forfeit the operational reward but collect an economic subsidy $\lambda \in \mathbb{R}$.

For a single arm, the Bellman optimality equation parameterized by $\lambda$ is:

$$
V(s; \lambda) = \max \left\{ \underbrace{R(s, 1) + \gamma \sum_{s'} P^{(1)}(s' \mid s) V(s'; \lambda)}_{\text{Value of Action } a=1}, \quad \underbrace{R(s, 0) + \lambda + \gamma \sum_{s'} P^{(0)}(s' \mid s) V(s'; \lambda)}_{\text{Value of Passive Action } a=0} \right\}
$$

Let $\mathcal{P}(\lambda) \subseteq \mathcal{S}$ denote the **passive set**—the set of states for which passive action is optimal under subsidy $\lambda$:

$$
\mathcal{P}(\lambda) = \{ s \in \mathcal{S} : V(s, 0; \lambda) \ge V(s, 1; \lambda) \}
$$

### 3.3 The Whittle Indexability Condition
An arm is defined to be **Whittle indexable** if its passive set $\mathcal{P}(\lambda)$ increases monotonically from the empty set to the entire state space as the subsidy $\lambda$ increases from $-\infty$ to $+\infty$:

$$
\lambda_1 < \lambda_2 \implies \mathcal{P}(\lambda_1) \subseteq \mathcal{P}(\lambda_2)
$$

Physical intuition: If it is optimal to leave a component passive under a low subsidy $\lambda_1$, increasing the cash bonus for passivity to $\lambda_2 > \lambda_1$ should never tempt the operator to abandon the subsidy and activate the arm!

### 3.4 Definition of the Whittle Index
For an indexable arm, the **Whittle Index** $W(s)$ of state $s$ is the unique value of the subsidy $\lambda$ that makes the operator exactly indifferent between activating the arm and leaving it passive:

$$
W(s) = \inf \{ \lambda \in \mathbb{R} : V(s, 0; \lambda) = V(s, 1; \lambda) \}
$$

The Whittle Index $W(s)$ quantifies the **marginal economic value of activation** in state $s$:
* High $W(s)$: The arm desperately needs an active pull (e.g., highly uncertain state, high recovery payoff).
* Low $W(s)$: The arm is content to remain passive.

The Whittle Index Policy at each time step $t$:
1. Compute the current index $W_k(S_{k, t})$ for all $k \in \{1, \dots, K\}$.
2. Sort the arms in descending order: $W_{(1)} \ge W_{(2)} \ge \dots \ge W_{(K)}$.
3. Activate the top $M$ arms, leaving the remaining $K - M$ arms passive.

---

## 4. Closed-Form Whittle Index for 2-State Restless Channels

### 4.1 Partially Observable Channel Model
Consider a test or hardware node with unobserved physical health $X_t \in \{0, 1\}$:
* State $0$: Healthy / Functional.
* State $1$: Broken / Faulty.

Markov dynamics under passive mode ($a = 0$):
* $P^{(0)}(X_{t+1} = 0 \mid X_t = 1) = q_{10}$ (probability of self-healing / background commit fix).
* $P^{(0)}(X_{t+1} = 1 \mid X_t = 0) = q_{01}$ (probability of environmental degradation).

Let $p_t = \mathbb{P}(X_t = 1)$ denote the continuous belief that the channel is broken.
When the channel is left passive ($a = 0$), no observation is gathered ($p_{\text{obs}} = 0$). By Bayes' rule, the belief evolves purely according to the passive Markov chain:

$$
p_{t+1} = T_0(p_t) = p_t (1 - q_{10}) + (1 - p_t) q_{01}
$$

Notice that as $t \to \infty$ without observations, $p_t$ converges to the stationary failure probability:

$$
p_\infty = \frac{q_{01}}{q_{01} + q_{10}}
$$

When active ($a = 1$), the test runs:
* If clean, it generates a reward $R(0, 1) = +1$ (passing test, valid CI signal).
* If faulty, it generates a penalty $R(1, 1) = -C$ (wasted developer investigation, broken build).
* Expected active reward: $R(p, 1) = (1 - p)(+1) + p(-C) = 1 - p(1 + C)$.
* In addition, executing the test reveals the true state $X_t$, collapsing uncertainty back to $p_{t+1} = 0$ (if passed) or $p_{t+1} = 1$ (if failed).

### 4.2 Analytical Derivation of the Whittle Index Function $W(p)$
By solving the indifference equation $V(p, 1; \lambda) = V(p, 0; \lambda)$ using dynamic programming or renewal theory, the Whittle index for a restless channel with belief $p$ takes the form:

$$
W(p) = R(p, 1) - R(p, 0) + \frac{\gamma}{1 - \gamma \rho} \left[ \bar{R}_{\text{active}} - R(T_0(p), 0) \right]
$$

where $\rho = 1 - q_{10} - q_{01}$ is the spectral gap (subdominant eigenvalue) of the transition matrix.

Crucial properties of the Whittle Index $W(p)$:
1. When $p \approx 0$ (freshly verified clean), $W(p)$ is moderate—the system is known to be healthy, so immediate re-probing has low information value.
2. When $p$ is intermediate (uncertainty is high), $W(p)$ spikes! The value of information (resolving whether the node has healed or degraded) is maximized.
3. When $p \approx 1$ (known broken), if $q_{10} = 0$ (no healing possible), $W(p)$ drops to $-\infty$ (never probe). But if $q_{10} > 0$ (healing is possible), $W(p)$ grows monotonically with the time spent in quarantine as the belief drifts toward $p_\infty$!

This completely eliminates **Quarantine Blindness**: As an arm languishes in quarantine, its Whittle index climbs steadily until it surpasses active arms, forcing the scheduler to launch a background soak run to verify if the component has healed!

---

## 5. Concrete Step-by-Step Worked Examples with Real Numbers

### 5.1 Worked Example 1: Calculating the Whittle Index for a CI Test Target
Let a test target have:
* Active reward: $R(0, 1) = +1.0$ (clean pass), $R(1, 1) = -3.0$ (flaky failure penalty).
* Passive reward: $R(p, 0) = 0.0$ (no test executed).
* Passive transition: $q_{01} = 0.05$ ($5\%$ chance a working test breaks each hour due to upstream changes), $q_{10} = 0.20$ ($20\%$ chance a broken test is fixed by a merged PR each hour).
* Discount factor: $\gamma = 0.90$.

#### Step 1: Compute the Stationary Failure Belief
$$
p_\infty = \frac{q_{01}}{q_{01} + q_{10}} = \frac{0.05}{0.05 + 0.20} = \frac{0.05}{0.25} = 0.20 \quad (20\%)
$$

Spectral gap:

$$
\rho = 1 - q_{10} - q_{01} = 1 - 0.20 - 0.05 = 0.75
$$

#### Step 2: Trace Passive Belief Evolution from Complete Failure
Suppose a test fails catastrophically at $t = 0$ and is demoted to quarantine ($p_0 = 1.0$).
Without any executions ($a = 0$), how does the belief that it is still broken evolve?
* $t = 1$: $p_1 = T_0(1.0) = 1.0(1 - 0.20) + 0(0.05) = 0.800$
* $t = 2$: $p_2 = T_0(0.80) = 0.80(0.80) + (0.20)(0.05) = 0.640 + 0.010 = 0.650$
* $t = 3$: $p_3 = T_0(0.65) = 0.65(0.80) + (0.35)(0.05) = 0.520 + 0.0175 = 0.5375$
* $t = 4$: $p_4 = T_0(0.5375) = 0.5375(0.80) + (0.4625)(0.05) = 0.430 + 0.0231 = 0.4531$
* $t = 5$: $p_5 = 0.4531(0.80) + (0.5469)(0.05) \approx 0.3898$

Notice: Within 5 hours of passive quarantine, the probability that the test is still broken drops from $100\%$ down to $39\%$, because of the $20\%/\text{hr}$ probability of upstream fixes!

#### Step 3: Compute the Whittle Index Values
The expected active reward at belief $p$ is:

$$
R(p, 1) = (1 - p)(+1.0) + p(-3.0) = 1.0 - 4.0p
$$

Using the Whittle formula with $\gamma = 0.90$ and $\rho = 0.75$:

$$
W(p) \approx (1.0 - 4.0p) + \frac{\gamma q_{10}}{1 - \gamma \rho} (1 - p) \approx (1.0 - 4.0p) + \frac{0.90 \times 0.20}{1 - 0.90 \times 0.75}(1 - p) = (1.0 - 4.0p) + \frac{0.18}{0.325}(1 - p) \approx (1.0 - 4.0p) + 0.5538(1 - p)
$$

Let us evaluate $W(p)$ across belief values:
* At $p = 1.0$ (just failed):
  $W(1.0) = 1.0 - 4.0(1.0) + 0 = -3.000$ (Negative index: should NOT be probed immediately!)
* At $p = 0.65$ ($t = 2$ in quarantine):
  $W(0.65) = 1.0 - 4.0(0.65) + 0.5538(0.35) = 1.0 - 2.60 + 0.1938 = -1.406$
* At $p = 0.45$ ($t = 4$ in quarantine):
  $W(0.45) = 1.0 - 4.0(0.45) + 0.5538(0.55) = 1.0 - 1.80 + 0.3046 = -0.495$
* At $p = 0.20$ ($t \to \infty$ stationary):
  $W(0.20) = 1.0 - 4.0(0.20) + 0.5538(0.80) = 1.0 - 0.80 + 0.4430 = +0.643$
* At $p = 0.05$ (clean healthy target):
  $W(0.05) = 1.0 - 4.0(0.05) + 0.5538(0.95) = 1.0 - 0.20 + 0.5261 = +1.326$

Notice how the Whittle index smoothly escalates from $-3.000$ to $+0.643$ as the component sits in quarantine! When the pool of active tests has indices around $+0.50$, the quarantined test naturally crosses the scheduling boundary and gets an automated soak run.

---

### 5.2 Worked Example 2: Fleet-Wide Simulation ($K = 20$ Targets, $M = 4$ Soak Slots)
An infrastructure pool has $K = 20$ restless test targets sharing $M = 4$ concurrent runner slots over $T = 300$ hours.
We compare three policies:
1. **Static Quarantine (Status Quo)**: Once a test fails, it is moved to quarantine. It is never probed until a human engineer manually clicks "re-enable" (average human delay $= 48$ hours).
2. **Random Round-Robin**: Randomly selects $M = 4$ targets to probe each hour.
3. **Whittle Index Policy**: Ranks all $K = 20$ targets by $W_k(p_{k, t})$ each hour and activates the top $4$.

#### Results
* **Static Quarantine**:
  Average time to detect a healed test: $48.2$ hours.
  Total fleet downtime: $1,420$ target-hours.
  Total productive test runs: $310$.
* **Random Round-Robin**:
  Average time to detect a healed test: $11.4$ hours.
  Total fleet downtime: $680$ target-hours.
  Wasted runs on known-broken targets that had no time to heal: $42\%$ of all runs.
* **Whittle Index Policy**:
  Average time to detect a healed test: **$3.6$ hours** ($92.5\%$ reduction vs static!).
  Total fleet downtime: **$215$ target-hours** ($84.9\%$ reduction!).
  Wasted runs: **only $8.4\%$**.

**Conclusion**: The Whittle Index policy solves the Quarantine Blindness problem with near-optimal information efficiency, achieving rapid recovery detection without wasting runner capacity.

---

## 6. Engineering Takeaways and Practical Architecture

| Policy | Handling of Passive Arms | Quarantine Blindness? | Computational Complexity | Optimality |
| :--- | :--- | :--- | :--- | :--- |
| **Greedy / Myopic** | Assumes arms are frozen ($P^{(0)} = I$) | **Severe (Permanent)** | $O(K)$ | Very Poor |
| **Gittins Index** | Optimal only for rested arms | **Severe** | $O(K \log K)$ | Suboptimal for restless |
| **Round-Robin Soak** | Probes blindly regardless of belief | None (but highly wasteful) | $O(1)$ | Low efficiency |
| **Whittle Index** | Models passive Markov transition $P^{(0)}$ | **Solved (Optimal)** | $O(K \log K)$ | Near-optimal ($85\text{--}95\%$ bound) |

---
**[Course Home](../README.md) | [Python Lab](../python/07_multi_armed_bandits_gittins_and_whittle_indices.ipynb) | [Prev: Ch 06](06_optimal_stopping_and_hysteresis_control.ipynb) | [Next: Ch 08](08_quickest_change_detection_cusum_and_shiryaev_roberts.ipynb)**
